# Week 3 Assignment: Image Segmentation of Handwritten Digits (PyTorch)

<img src='https://storage.googleapis.com/tensorflow-3-public/assets/images/m2nist_segmentation.png' alt='m2nist digits'>

In this week's assignment, you will build a model that predicts the segmentation masks (pixel-wise label map) of handwritten digits. This model will be trained on the [M2NIST dataset](https://www.kaggle.com/farhanhubble/multimnistm2nist), a multi digit MNIST. If you've done the ungraded lab on the CamVid dataset, then many of the steps here will look familiar.

You will build a Convolutional Neural Network (CNN) from scratch for the downsampling path and use a Fully Convolutional Network, FCN-8, to upsample and produce the pixel-wise label map. The model will be evaluated using the intersection over union (IOU) and Dice Score.

> This is a PyTorch port of the original TensorFlow assignment. The exercises are the same, but the building blocks are `nn.Module`s instead of Keras layers, and the training loop is written out explicitly. The Coursera autograder expects a Keras model file, so the PyTorch model cannot be submitted for grading; the notebook computes the same grade locally instead.

## Exercises

We've given you some boilerplate code to work with and these are the 5 exercises you need to fill out before you can successfully get the segmentation masks.

* [Exercise 1 - Define the Basic Convolution Block](#exercise-1)
* [Exercise 2 - Define the Downsampling Path](#exercise-2)
* [Exercise 3 - Define the FCN-8 decoder](#exercise-3)
* [Exercise 4 - Configure the Model for Training](#exercise-4)
* [Exercise 5 - Model Training](#exercise-5)

## Imports

As usual, let's start by importing the packages you will use in this lab.

In [ ]:
import os
import zipfile
import platform
import urllib.request

import PIL.Image, PIL.ImageFont, PIL.ImageDraw
import numpy as np
from matplotlib import pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Download the dataset

[M2NIST](https://www.kaggle.com/farhanhubble/multimnistm2nist) is a **multi digit** [MNIST](http://yann.lecun.com/exdb/mnist/).
Each image has up to 3 digits from MNIST digits and the corresponding labels file has the segmentation masks.

The dataset is available on [Kaggle](https://www.kaggle.com) and you can find it [here](https://www.kaggle.com/farhanhubble/multimnistm2nist)

To make it easier for you, we're hosting it on Google Cloud so you can download without Kaggle credentials.

In [ ]:
# download zipped dataset
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/m2nist.zip"):
    urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/m2nist.zip", "data/m2nist.zip")

# find and extract to a local folder ('data/m2nist')
local_zip = 'data/m2nist.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('data/m2nist')
zip_ref.close()

## Load and Preprocess the Dataset

This dataset can be easily preprocessed since it is available as **Numpy Array Files (.npy)**

1. **combined.npy** has the image files containing the multiple MNIST digits. Each image is of size **64 x 84** (height x width, in pixels).

2. **segmented.npy** has the corresponding segmentation masks. Each segmentation mask is also of size **64 x 84**, one-hot encoded over the 11 classes (the digits 0 to 9 plus the background).

This dataset has **5000** samples and you can make appropriate training, validation, and test splits as required for the problem.

With that, let's define a few utility functions for loading and preprocessing the dataset. Note that `nn.CrossEntropyLoss` expects a label map of integer class ids rather than one-hot vectors, so the one-hot masks are converted with `argmax` when they are loaded.

In [ ]:
BATCH_SIZE = 32

# DataLoader worker processes on macOS are started with "spawn", which cannot see classes defined
# inside a notebook (such as the Dataset below). "fork" works fine for the work the loaders do here.
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None


def read_image_and_annotation(image, annotation):
  '''
  Casts the image and annotation to their expected data type and
  normalizes the input image so that each pixel is in the range [-1, 1]

  Args:
    image (numpy array) -- input image, shape (64, 84)
    annotation (numpy array) -- ground truth label map, shape (64, 84)

  Returns:
    preprocessed image-annotation pair: image tensor (1, 64, 84), annotation tensor (64, 84)
  '''

  image = torch.from_numpy(np.asarray(image, dtype=np.float32)).unsqueeze(0)   # add the channel axis
  annotation = torch.from_numpy(np.asarray(annotation)).long()
  image = image / 127.5
  image -= 1

  return image, annotation


class M2NISTDataset(Dataset):
  '''pairs of (preprocessed image, label map)'''

  def __init__(self, images, annos):
    '''
    Stores the images and label maps this split will serve.

    Args:
      images (array) -- M2NIST images, shape (N, 64, 84)
      annos (array) -- matching label maps of class ids, shape (N, 64, 84)
    '''
    self.images = images
    self.annos = annos

  def __len__(self):
    '''
    Reports how many image and label-map pairs this split holds.

    Returns:
      int -- number of image and label-map pairs
    '''
    return len(self.images)

  def __getitem__(self, idx):
    '''
    Fetches pair `idx` and normalizes the image to [-1, 1].

    Args:
      idx (int) -- index of the pair to fetch

    Returns:
      (tensor, tensor) -- normalized image (1, 64, 84) and its label map (64, 84)
    '''
    return read_image_and_annotation(self.images[idx], self.annos[idx])


def get_training_dataset(images, annos):
  '''
  Prepares shuffled batches of the training set.

  Args:
    images (numpy array) -- images of the train set
    annos (numpy array) -- label maps of the train set

  Returns:
    DataLoader containing the preprocessed train set
  '''
  training_dataset = DataLoader(M2NISTDataset(images, annos), batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=2, multiprocessing_context=MP_CONTEXT, persistent_workers=True)

  return training_dataset


def get_validation_dataset(images, annos):
  '''
  Prepares batches of the validation set.

  Args:
    images (numpy array) -- images of the val set
    annos (numpy array) -- label maps of the val set

  Returns:
    DataLoader containing the preprocessed validation set
  '''
  validation_dataset = DataLoader(M2NISTDataset(images, annos), batch_size=BATCH_SIZE)

  return validation_dataset


def get_test_dataset(images, annos):
  '''
  Prepares batches of the test set.

  Args:
    images (numpy array) -- images of the test set
    annos (numpy array) -- label maps of the test set

  Returns:
    DataLoader containing the preprocessed test set
  '''
  test_dataset = DataLoader(M2NISTDataset(images, annos), batch_size=BATCH_SIZE, drop_last=True)

  return test_dataset


def load_images_and_segments():
  '''
  Loads the images and segments as numpy arrays from npy files
  and makes splits for training, validation and test datasets.

  Returns:
    3 tuples containing the train, val, and test splits
  '''

  #Loads images and segmentation masks.
  images = np.load('data/m2nist/combined.npy')
  # the masks are one-hot encoded (5000, 64, 84, 11); keep the class id of each pixel instead
  segments = np.argmax(np.load('data/m2nist/segmented.npy'), axis=-1).astype(np.uint8)

  #Makes training, validation, test splits from loaded images and segmentation masks.
  train_images, val_images, train_annos, val_annos = train_test_split(images, segments, test_size=0.2, shuffle=True)
  val_images, test_images, val_annos, test_annos = train_test_split(val_images, val_annos, test_size=0.2, shuffle=True)

  return (train_images, train_annos), (val_images, val_annos), (test_images, test_annos)

You can now load the preprocessed dataset and define the training, validation, and test sets.

In [ ]:
# Load Dataset
train_slices, val_slices, test_slices = load_images_and_segments()

# Create training, validation, test datasets.
training_dataset = get_training_dataset(train_slices[0], train_slices[1])
validation_dataset = get_validation_dataset(val_slices[0], val_slices[1])
test_dataset = get_test_dataset(test_slices[0], test_slices[1])

## Let's Take a Look at the Dataset

You may want to visually inspect the dataset before and after training. Like above, we've included utility functions to help show a few images as well as their annotations (i.e. labels).

In [ ]:
# Visualization Utilities

# there are 11 classes in the dataset: one class for each digit (0 to 9) plus the background class
n_classes = 11

# assign a random color for each class
colors = [tuple(np.random.randint(256, size=3) / 255.0) for i in range(n_classes)]

def fuse_with_pil(images):
  '''
  Creates a blank image and pastes input images

  Args:
    images (list of numpy arrays) - numpy array representations of the images to paste

  Returns:
    PIL Image object containing the images
  '''

  widths = (image.shape[1] for image in images)
  heights = (image.shape[0] for image in images)
  total_width = sum(widths)
  max_height = max(heights)

  new_im = PIL.Image.new('RGB', (total_width, max_height))

  x_offset = 0
  for im in images:
    pil_image = PIL.Image.fromarray(np.uint8(im))
    new_im.paste(pil_image, (x_offset,0))
    x_offset += im.shape[1]

  return new_im


def give_color_to_annotation(annotation):
  '''
  Converts a 2-D annotation to a numpy array with shape (height, width, 3) where
  the third axis represents the color channel. The label values are multiplied by
  255 and placed in this axis to give color to the annotation

  Args:
    annotation (numpy array) - label map array

  Returns:
    the annotation array with an additional color channel/axis
  '''
  annotation = np.asarray(annotation)
  seg_img = np.zeros( (annotation.shape[0],annotation.shape[1], 3) ).astype('float')

  for c in range(n_classes):
    segc = (annotation == c)
    seg_img[:,:,0] += segc*( colors[c][0] * 255.0)
    seg_img[:,:,1] += segc*( colors[c][1] * 255.0)
    seg_img[:,:,2] += segc*( colors[c][2] * 255.0)

  return seg_img


def image_to_uint8(image):
  '''
  Converts a normalized image tensor back to something matplotlib can show.

  Args:
    image (tensor) -- image in the range [-1, 1], shape (1, height, width)

  Returns:
    array -- uint8 image of shape (height, width)
  '''
  image = np.asarray(image)
  image = image + 1
  image = image * 127.5
  image = np.reshape(image, (image.shape[-2], image.shape[-1],))
  return np.uint8(image)


def show_annotation_and_prediction(image, annotation, prediction, iou_list, dice_score_list):
  '''
  Displays the images with the ground truth and predicted label maps. Also overlays the metrics.

  Args:
    image (tensor) -- the input image, shape (1, height, width)
    annotation (numpy array) -- the ground truth label map
    prediction (numpy array) -- the predicted label map
    iou_list (list of floats) -- the IOU values for each class
    dice_score_list (list of floats) -- the Dice Score for each class
  '''

  true_img = give_color_to_annotation(annotation)
  pred_img = give_color_to_annotation(prediction)

  image = image_to_uint8(image)
  images = [image, np.uint8(pred_img), np.uint8(true_img)]

  metrics_by_id = [(idx, iou, dice_score) for idx, (iou, dice_score) in enumerate(zip(iou_list, dice_score_list)) if iou > 0.0 and idx < 10]
  metrics_by_id.sort(key=lambda tup: tup[1], reverse=True)  # sorts in place

  display_string_list = ["{}: IOU: {} Dice Score: {}".format(idx, iou, dice_score) for idx, iou, dice_score in metrics_by_id]
  display_string = "\n".join(display_string_list)

  plt.figure(figsize=(15, 4))

  for idx, im in enumerate(images):
    plt.subplot(1, 3, idx+1)
    if idx == 1:
      plt.xlabel(display_string)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(im)


def show_annotation_and_image(image, annotation):
  '''
  Displays the image and its annotation side by side

  Args:
    image (tensor) -- the input image, shape (1, height, width)
    annotation (tensor) -- the label map, shape (height, width)
  '''
  seg_img = give_color_to_annotation(annotation)

  image = image_to_uint8(image)
  images = [image, seg_img]

  fused_img = fuse_with_pil(images)
  plt.imshow(fused_img)


def list_show_annotation(dataloader, num_images):
  '''
  Displays images and its annotations side by side

  Args:
    dataloader (DataLoader) -- batches of images and annotations
    num_images (int) -- number of images to display
  '''
  ds = dataloader.dataset

  plt.figure(figsize=(20, 15))
  plt.title("Images And Annotations")
  plt.subplots_adjust(bottom=0.1, top=0.9, hspace=0.05)

  for idx in range(num_images):
    image, annotation = ds[idx]
    plt.subplot(5, 5, idx + 1)
    plt.yticks([])
    plt.xticks([])
    show_annotation_and_image(image.numpy(), annotation.numpy())

You can view a subset of the images from the dataset with the `list_show_annotation()` function defined above. Run the cells below to see the image on the left and its pixel-wise ground truth label map on the right.

In [ ]:
# get 10 images from the training set
list_show_annotation(training_dataset, 10)

In [ ]:
# get 10 images from the validation set
list_show_annotation(validation_dataset, 10)

You see from the images above the colors assigned to each class (i.e 0 to 9 plus the background). If you don't like these colors, feel free to rerun the cell where `colors` is defined to get another set of random colors. Alternatively, you can assign the RGB values for each class instead of relying on random values.

## Define the Model

As discussed in the lectures, the image segmentation model will have two paths:

1. **Downsampling Path** - This part of the network extracts the features in the image. This is done through a series of convolution and pooling layers. The final output is a reduced image (because of the pooling layers) with the extracted features. You will build a custom CNN from scratch for this path.

2. **Upsampling Path** - This takes the output of the downsampling path and generates the predictions while also converting the image back to its original size. You will use an FCN-8 decoder for this path.

### Define the Basic Convolution Block

<a name='exercise-1'></a>

#### **Exercise 1**

Please complete the function below to build the basic convolution block for our CNN. This will have two [Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) layers each followed by a [LeakyReLU](https://pytorch.org/docs/stable/generated/torch.nn.LeakyReLU.html), then [max pooled](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [batch-normalized](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html). Use `nn.Sequential` to stack these layers.

$$Input -> Conv2d -> LeakyReLU -> Conv2d -> LeakyReLU -> MaxPool2d -> BatchNorm2d$$

When defining the `Conv2d` layers, note that PyTorch always keeps the 'channels' dimension first (`(batch, channels, height, width)`), so each layer needs the number of input channels (`in_channels` for the first convolution, `filters` for the second). Take note of the `padding` argument too like you did in the ungraded labs (`padding='same'` keeps the height and width unchanged). `BatchNorm2d` needs the number of channels it normalizes.

In [ ]:
def conv_block(in_channels, filters, kernel_size, pooling_size, pool_strides):
  '''
  Builds the basic convolution block used throughout the encoder (Exercise 1).

  Args:
    in_channels (int) -- number of channels of the block's input
    filters (int) -- number of filters of the Conv2d layers
    kernel_size (int) -- kernel_size setting of the Conv2d layers
    pooling_size (int) -- pooling size of the MaxPool2d layer
    pool_strides (int) -- strides setting of the MaxPool2d layer

  Returns:
    (nn.Sequential) module producing the max pooled and batch-normalized features of the input
  '''
  ### START CODE HERE ###
  # use nn.Sequential to stack the layers as shown in the diagram above
  block = nn.Sequential(
      # padding='same' keeps the height and width unchanged, like Keras' padding='same'
      nn.Conv2d(in_channels, filters, kernel_size, padding='same'),   # Conv2d
      nn.LeakyReLU(),                                                 # LeakyReLU
      nn.Conv2d(filters, filters, kernel_size, padding='same'),       # Conv2d
      nn.LeakyReLU(),                                                 # LeakyReLU
      nn.MaxPool2d(pooling_size, pool_strides),                       # MaxPool2d
      nn.BatchNorm2d(filters),                                        # BatchNorm2d
  )
  ### END CODE HERE ###

  return block

In [ ]:
# TEST CODE:

test_block = conv_block(1, 32, 3, 2, 2)

print(summary(test_block, input_size=(1, 64, 84), batch_dim=0, device="cpu"))

# free up test resources
del test_block

**Expected Output**:

Please pay attention to the layer types and the *Output Shape* column (PyTorch shows shapes as `[batch, channels, height, width]`).

```txt
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
Sequential                               [1, 32, 32, 42]           --
├─Conv2d: 1-1                            [1, 32, 64, 84]           320
├─LeakyReLU: 1-2                         [1, 32, 64, 84]           --
├─Conv2d: 1-3                            [1, 32, 64, 84]           9,248
├─LeakyReLU: 1-4                         [1, 32, 64, 84]           --
├─MaxPool2d: 1-5                         [1, 32, 32, 42]           --
├─BatchNorm2d: 1-6                       [1, 32, 32, 42]           64
==========================================================================================
Total params: 9,632
Trainable params: 9,632
Non-trainable params: 0
```

(Keras counted the batch normalization's moving mean and variance as 64 extra non-trainable parameters; PyTorch stores them as buffers, so they do not show up in the parameter count.)

### Define the Downsampling Path

<a name='exercise-2'></a>

#### **Exercise 2**

Now that we've defined the building block of our encoder, you can now build the downsampling path. Please complete the module below to create the encoder. This should chain together five convolution building blocks to create a feature extraction CNN minus the fully connected layers. The blocks use 32, 64, 128, 256 and 256 filters, a kernel size of 3, and (as recommended in note 2 below) a pool size and stride of 2.

*Notes*:
1. To optimize processing or to make the output dimensions of each layer easier to work with, it is sometimes advisable to apply some zero-padding to the input image. With the boilerplate code we have provided below, we suggest padding the input width to 96 pixels using [nn.ZeroPad2d](https://pytorch.org/docs/stable/generated/torch.nn.ZeroPad2d.html) (its `padding` argument is `(left, right, top, bottom)`, so adding 12 columns on the right is `(0, 12, 0, 0)`). This works well if you're going to use the first ungraded lab of this week as reference. This is not required however. You can remove it later and see how it will affect your parameters. For instance, you might need to pass in a non-square kernel size to the decoder in Exercise 3 (e.g. `(4,5)`) to match the output dimensions of Exercise 2.

2. We recommend keeping the pool size and stride parameters constant at 2.

In [ ]:
class FCN8Encoder(nn.Module):
    '''
    Defines the downsampling path of the image segmentation model.

    The forward pass takes a batch of images of shape (batch, 1, 64, 84) and returns
    a tuple of tensors: the features extracted at blocks 3 to 5
    '''

    def __init__(self):
        '''
        Builds the zero padding and the five convolution blocks of the downsampling path (Exercise 2).
        '''
        super().__init__()

        ### START CODE HERE ###

        # pad the input image width to 96 pixels: (0, 12, 0, 0) adds 12 columns on the right
        self.pad = nn.ZeroPad2d((0, 12, 0, 0))

        # Block 1                        shape: (N, 1, 64, 96) -> (N, 32, 32, 48)
        self.block1 = conv_block(1, 32, 3, 2, 2)

        # Block 2                        shape: -> (N, 64, 16, 24)
        self.block2 = conv_block(32, 64, 3, 2, 2)

        # Block 3                        shape: -> (N, 128, 8, 12)
        self.block3 = conv_block(64, 128, 3, 2, 2)

        # Block 4                        shape: -> (N, 256, 4, 6)
        self.block4 = conv_block(128, 256, 3, 2, 2)

        # Block 5                        shape: -> (N, 256, 2, 3)
        self.block5 = conv_block(256, 256, 3, 2, 2)

        ### END CODE HERE ###

    def forward(self, img_input):
        '''
        Runs the image down through the blocks, keeping the features the decoder needs.

        Args:
          img_input (tensor) -- batch of images, shape (N, 1, 64, 84)

        Returns:
          tuple of tensors -- features saved at blocks 3, 4 and 5
        '''
        ### START CODE HERE ###
        # pad the input image width to 96 pixels
        x = self.pad(img_input)

        # Block 1
        x = self.block1(x)

        # Block 2
        x = self.block2(x)

        # Block 3
        x = self.block3(x)
        # save the feature map at this stage
        f3 = x

        # Block 4
        x = self.block4(x)
        # save the feature map at this stage
        f4 = x

        # Block 5
        x = self.block5(x)
        # save the feature map at this stage
        f5 = x

        ### END CODE HERE ###

        return (f3, f4, f5)

In [ ]:
# TEST CODE:

test_encoder = FCN8Encoder()

print(summary(test_encoder, input_size=(1, 64, 84), batch_dim=0, device="cpu", depth=2))

del test_encoder

**Expected Output**:

You should see the layers of your `conv_block()` being repeated 5 times like the output below.

```txt
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
FCN8Encoder                              [1, 128, 8, 12]           --
├─ZeroPad2d: 1-1                         [1, 1, 64, 96]            --
├─Sequential: 1-2                        [1, 32, 32, 48]           --
│    └─Conv2d: 2-1                       [1, 32, 64, 96]           320
│    └─LeakyReLU: 2-2                    [1, 32, 64, 96]           --
│    └─Conv2d: 2-3                       [1, 32, 64, 96]           9,248
│    └─LeakyReLU: 2-4                    [1, 32, 64, 96]           --
│    └─MaxPool2d: 2-5                    [1, 32, 32, 48]           --
│    └─BatchNorm2d: 2-6                  [1, 32, 32, 48]           64
├─Sequential: 1-3                        [1, 64, 16, 24]           --
│    └─Conv2d: 2-7                       [1, 64, 32, 48]           18,496
│    └─LeakyReLU: 2-8                    [1, 64, 32, 48]           --
│    └─Conv2d: 2-9                       [1, 64, 32, 48]           36,928
│    └─LeakyReLU: 2-10                   [1, 64, 32, 48]           --
│    └─MaxPool2d: 2-11                   [1, 64, 16, 24]           --
│    └─BatchNorm2d: 2-12                 [1, 64, 16, 24]           128
├─Sequential: 1-4                        [1, 128, 8, 12]           --
│    └─Conv2d: 2-13                      [1, 128, 16, 24]          73,856
│    └─LeakyReLU: 2-14                   [1, 128, 16, 24]          --
│    └─Conv2d: 2-15                      [1, 128, 16, 24]          147,584
│    └─LeakyReLU: 2-16                   [1, 128, 16, 24]          --
│    └─MaxPool2d: 2-17                   [1, 128, 8, 12]           --
│    └─BatchNorm2d: 2-18                 [1, 128, 8, 12]           256
├─Sequential: 1-5                        [1, 256, 4, 6]            --
│    └─Conv2d: 2-19                      [1, 256, 8, 12]           295,168
│    └─LeakyReLU: 2-20                   [1, 256, 8, 12]           --
│    └─Conv2d: 2-21                      [1, 256, 8, 12]           590,080
│    └─LeakyReLU: 2-22                   [1, 256, 8, 12]           --
│    └─MaxPool2d: 2-23                   [1, 256, 4, 6]            --
│    └─BatchNorm2d: 2-24                 [1, 256, 4, 6]            512
├─Sequential: 1-6                        [1, 256, 2, 3]            --
│    └─Conv2d: 2-25                      [1, 256, 4, 6]            590,080
│    └─LeakyReLU: 2-26                   [1, 256, 4, 6]            --
│    └─Conv2d: 2-27                      [1, 256, 4, 6]            590,080
│    └─LeakyReLU: 2-28                   [1, 256, 4, 6]            --
│    └─MaxPool2d: 2-29                   [1, 256, 2, 3]            --
│    └─BatchNorm2d: 2-30                 [1, 256, 2, 3]            512
==========================================================================================
Total params: 2,353,312
Trainable params: 2,353,312
Non-trainable params: 0
```

### Define the FCN-8 decoder

<a name='exercise-3'></a>

#### **Exercise 3**

Now you can define the upsampling path taking the outputs of convolutions at each stage as arguments. This will be very similar to what you did in the ungraded lab (VGG16-FCN8-CamVid) so you can refer to it if you need a refresher.
* Note: the layers are created in `__init__` and used in `forward`. Each `Conv2d`/`ConvTranspose2d` needs its number of input channels: the pool 4 features have 256 channels and the pool 3 features have 128 channels (see the expected output of Exercise 2).
* `Cropping2D(cropping=(1,1))` in Keras is simply slicing one pixel off each side of the height and width: `o[:, :, 1:-1, 1:-1]`.
* The Keras version ended with an activation layer. Here the decoder returns the raw class scores (logits) because `nn.CrossEntropyLoss` applies the softmax itself.

Here is also the diagram you saw in class on how it should work:

<img src='https://drive.google.com/uc?export=view&id=1lrqB4YegV8jXWNfyYAaeuFlwXIc54aRP' alt='fcn-8'>

In [ ]:
class FCN8Decoder(nn.Module):
  '''
  Defines the FCN 8 decoder.

  Args:
    n_classes (int) -- number of classes

  The forward pass takes the tuple of encoder features (f3, f4, f5) and returns a tensor
  of shape (batch, n_classes, 64, 84) with the class scores of every pixel
  '''

  def __init__(self, n_classes):
    '''
    Builds the convolution and transposed convolution layers of the upsampling path (Exercise 3).

    Args:
      n_classes (int) -- number of segmentation classes
    '''
    super().__init__()

    # number of filters
    n = 512

    # convolutional layers on top of the CNN extractor.
    self.conv6 = nn.Sequential(nn.Conv2d(256, n, kernel_size=(7, 7), padding='same'), nn.ReLU(), nn.Dropout(0.5))
    self.conv7 = nn.Sequential(nn.Conv2d(n, n, kernel_size=(1, 1), padding='same'), nn.ReLU(), nn.Dropout(0.5))
    self.score = nn.Sequential(nn.Conv2d(n, n_classes, kernel_size=(1, 1), padding='same'), nn.ReLU())

    ### START CODE HERE ###

    # transposed convolution that upsamples the n_classes channels 2x (kernel size 4, stride 2, no bias).
    # k=4, s=2 overshoots by one pixel each side, which the crop in forward() removes.
    self.upsample_1 = nn.ConvTranspose2d(n_classes, n_classes, 4, 2, bias=False)

    # 1x1 convolution (followed by a ReLU) that turns the pool 4 features into n_classes channels
    self.pool4_conv = nn.Sequential(nn.Conv2d(256, n_classes, 1, padding='same'), nn.ReLU())

    # transposed convolution that upsamples 2x again (kernel size 4, stride 2, no bias)
    self.upsample_2 = nn.ConvTranspose2d(n_classes, n_classes, 4, 2, bias=False)

    # 1x1 convolution (followed by a ReLU) that turns the pool 3 features into n_classes channels
    self.pool3_conv = nn.Sequential(nn.Conv2d(128, n_classes, 1, padding='same'), nn.ReLU())

    # transposed convolution that upsamples up to the size of the original image (kernel size 8, stride 8, no bias)
    self.upsample_3 = nn.ConvTranspose2d(n_classes, n_classes, 8, 8, bias=False)

    ### END CODE HERE ###

  def forward(self, convs):
    '''
    Upsamples back to the input resolution, adding in the skip connections.

    Args:
      convs (tuple) -- the encoder features (f3, f4, f5)

    Returns:
      tensor -- per-pixel class scores, shape (N, n_classes, 64, 84)
    '''
    # features from the encoder stage
    f3, f4, f5 = convs

    o = self.conv6(f5)
    o = self.conv7(o)
    o = self.score(o)

    ### START CODE HERE ###

    # Upsample `o` above and crop any extra pixels introduced
    o = self.upsample_1(o)               # shape: (N, C, 2, 3) -> (N, C, 6, 8)
    o = o[:, :, 1:-1, 1:-1]              # shape: -> (N, C, 4, 6), matching f4

    # load the pool 4 prediction and do a 1x1 convolution to reshape it to the same shape of `o` above
    o2 = f4
    o2 = self.pool4_conv(o2)

    # add the results of the upsampling and pool 4 prediction
    o = o + o2

    # upsample the resulting tensor of the operation you just did
    o = self.upsample_2(o)               # shape: -> (N, C, 10, 14)
    o = o[:, :, 1:-1, 1:-1]              # shape: -> (N, C, 8, 12), matching f3

    # load the pool 3 prediction and do a 1x1 convolution to reshape it to the same shape of `o` above
    o2 = f3
    o2 = self.pool3_conv(o2)

    # add the results of the upsampling and pool 3 prediction
    o = o + o2

    # upsample up to the size of the original image
    o = self.upsample_3(o)               # shape: -> (N, C, 64, 96)

    ### END CODE HERE ###

    # crop the padded width (96) back to the width of the original image (84)
    o = o[:, :, :, :84]

    return o

In [ ]:
# TEST CODE

test_encoder = FCN8Encoder()
test_fcn8_decoder = FCN8Decoder(11)

with torch.no_grad():
  test_output = test_fcn8_decoder(test_encoder(torch.zeros(1, 1, 64, 84)))
print(test_output.shape)

del test_encoder, test_fcn8_decoder, test_output

**Expected Output:**

```txt
torch.Size([1, 11, 64, 84])
```

### Define the Complete Model

The downsampling and upsampling paths can now be combined as shown below.

In [ ]:
class SegmentationModel(nn.Module):
  '''
  M2NIST segmentation model: the FCN-8 encoder chained to the FCN-8 decoder.
  '''
  def __init__(self, n_classes):
    '''
    Builds the encoder and the decoder.

    Args:
      n_classes (int) -- number of segmentation classes
    '''
    super().__init__()
    # start the encoder using the default input size 64 x 84
    self.encoder = FCN8Encoder()
    # the decoder that turns the convolutions obtained in the encoder into the label map
    self.decoder = FCN8Decoder(n_classes)

  def forward(self, img_input):
    '''
    Runs the image through the encoder and then the decoder.

    Args:
      img_input (tensor) -- batch of images, shape (N, 1, 64, 84)

    Returns:
      tensor -- per-pixel class scores, shape (N, n_classes, 64, 84)
    '''
    convs = self.encoder(img_input)
    dec_op = self.decoder(convs)
    return dec_op

# define the model and move it to the device
model = SegmentationModel(n_classes).to(device)

In [ ]:
summary(model, input_size=(1, 64, 84), batch_dim=0, device=device, depth=2)

## Configure the Model for Training

<a name='exercise-4'></a>

### **Exercise 4**

Define an appropriate loss function and optimizer (this is what `model.compile()` did in Keras). Remember that the label maps contain integer class ids for every pixel and the model outputs logits. The accuracy metric is computed for you in the training loop below.

In [ ]:
### START CODE HERE ###
# each pixel gets one of n_classes labels, so this is multi-class classification per pixel.
# CrossEntropyLoss applies the softmax itself, which is why the decoder returns raw logits.
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
### END CODE HERE ###

## Model Training

<a name='exercise-5'></a>

### **Exercise 5**

You can now train the model. Set the number of epochs and observe the metrics returned at each iteration. You can also terminate the cell execution if you think your model is performing well already.

In [ ]:
# OTHER THAN SETTING THE EPOCHS NUMBER, DO NOT CHANGE ANY OTHER CODE

### START CODE HERE ###
EPOCHS = 70
### END CODE HERE ###

steps_per_epoch = 4000//BATCH_SIZE
validation_steps = 800//BATCH_SIZE
test_steps = 200//BATCH_SIZE


def run_epoch(loader, model, loss_fn, optimizer, device, train, steps):
  '''
  Runs `steps` batches from `loader`, training or evaluating.

  Args:
    loader (DataLoader) -- yields (images, label maps) batches
    model (nn.Module) -- the segmentation model
    loss_fn (callable) -- loss applied to (logits, label maps)
    optimizer (Optimizer) -- updates weights; only used when train is True
    device (torch.device) -- device the batches are moved to
    train (bool) -- True updates the weights, False only measures
    steps (int) -- number of batches to run

  Returns:
    (float, float) -- mean loss and mean per-pixel accuracy
  '''
  model.train(train)
  total_loss, correct, count = 0.0, 0.0, 0
  with torch.set_grad_enabled(train):
    for step, (images, annotations) in enumerate(loader):
      if step >= steps:
        break
      images, annotations = images.to(device), annotations.to(device)
      logits = model(images)
      loss = loss_fn(logits, annotations)
      if train:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
      total_loss += loss.item() * len(images)
      correct += (logits.argmax(1) == annotations).float().mean().item() * len(images)
      count += len(images)
  return total_loss / count, correct / count


history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
for epoch in range(EPOCHS):
  train_loss, train_acc = run_epoch(training_dataset, model, loss_fn, optimizer, device, train=True, steps=steps_per_epoch)
  val_loss, val_acc = run_epoch(validation_dataset, model, loss_fn, optimizer, device, train=False, steps=validation_steps)
  history['loss'].append(train_loss); history['accuracy'].append(train_acc)
  history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
  print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

**Expected Output:**

The losses should generally be decreasing and the accuracies should generally be increasing. For example, observing the first 4 epochs should output something similar:

```txt
Epoch 1/70 - loss: 0.5542 - accuracy: 0.8635 - val_loss: 0.5335 - val_accuracy: 0.9427
Epoch 2/70 - loss: 0.2315 - accuracy: 0.9425 - val_loss: 0.3362 - val_accuracy: 0.9427
Epoch 3/70 - loss: 0.2118 - accuracy: 0.9426 - val_loss: 0.2592 - val_accuracy: 0.9427
Epoch 4/70 - loss: 0.1782 - accuracy: 0.9431 - val_loss: 0.1770 - val_accuracy: 0.9432
```

## Model Evaluation

### Make Predictions

Let's get the predictions using our test dataset as input and print the shape. The model outputs logits, so a `softmax` over the class axis turns them into probabilities.

In [ ]:
model.eval()
results = []
with torch.no_grad():
  for step, (images, _) in enumerate(test_dataset):
    if step >= test_steps:
      break
    results.append(torch.softmax(model(images.to(device)), dim=1).cpu())
results = torch.cat(results).numpy()

print(results.shape)

As you can see, the resulting shape is `(192, 11, 64, 84)`. This means that for each of the 192 images that we have in our test set, there are 11 predictions generated (i.e. one for each class: 0 to 9 plus background). Note that PyTorch keeps the class axis right after the batch axis.

Thus, if you want to see the *probability* of the upper leftmost pixel of the 1st image belonging to class 0, then you can print something like `results[0,0,0,0]`. If you want the probability of the same pixel at class 10, then do `results[0,10,0,0]`.

In [ ]:
print(results[0,0,0,0])
print(results[0,10,0,0])

What we're interested in is to get the *index* of the highest probability of each of these 11 slices and combine them in a single image. We can do that by getting the [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) at this axis.

In [ ]:
results = np.argmax(results, axis=1)

print(results.shape)

The new array generated per image now only specifies the indices of the class with the highest probability. Let's see the output class of the upper most left pixel. As you might have observed earlier when you inspected the dataset, the upper left corner is usually just part of the background (class 10). The actual digits are written somewhere in the middle parts of the image.

In [ ]:
print(results[0,0,0])

# prediction map for image 0
print(results[0,:,:])

We will use this `results` array when we evaluate our predictions.

### Metrics

We showed in the lectures two ways to evaluate your predictions. The *intersection over union (IOU)* and the *dice score*. Recall that:

$$IOU = \frac{area\_of\_overlap}{area\_of\_union}$$
<br>
$$Dice Score = 2 * \frac{area\_of\_overlap}{combined\_area}$$

The code below does that for you as you've also seen in the ungraded lab. A small smoothing factor is introduced in the denominators to prevent possible division by zero.

In [ ]:
def class_wise_metrics(y_true, y_pred):
  '''
  Computes the class-wise IOU and Dice Score.

  Args:
    y_true (array) -- ground truth label maps
    y_pred (array) -- predicted label maps of the same shape

  Returns:
    (list, list) -- IOU and Dice score, one entry per class
  '''
  class_wise_iou = []
  class_wise_dice_score = []

  smoothing_factor = 0.00001

  for i in range(n_classes):
    intersection = np.sum((y_pred == i) * (y_true == i))
    y_true_area = np.sum((y_true == i))
    y_pred_area = np.sum((y_pred == i))
    combined_area = y_true_area + y_pred_area

    iou = (intersection) / (combined_area - intersection + smoothing_factor)
    class_wise_iou.append(iou)

    dice_score =  2 * ((intersection) / (combined_area + smoothing_factor))
    class_wise_dice_score.append(dice_score)

  return class_wise_iou, class_wise_dice_score

### Visualize Predictions

In [ ]:
# place a number here between 0 to 191 to pick an image from the test set
integer_slider = 105

# gather the test images and ground truth label maps that the predictions were made on
images = []
y_true_segments = []
for step, (image, annotation) in enumerate(test_dataset):
  if step >= test_steps:
    break
  images.append(image)
  y_true_segments.append(annotation)
images = torch.cat(images)
y_true_segments = torch.cat(y_true_segments).numpy()


iou, dice_score = class_wise_metrics(y_true_segments[integer_slider], results[integer_slider])
show_annotation_and_prediction(images[integer_slider], y_true_segments[integer_slider], results[integer_slider], iou, dice_score)

### Compute IOU Score and Dice Score of your model

In [ ]:
cls_wise_iou, cls_wise_dice_score = class_wise_metrics(y_true_segments, results)

average_iou = 0.0
for idx, (iou, dice_score) in enumerate(zip(cls_wise_iou[:-1], cls_wise_dice_score[:-1])):
  print("Digit {}: IOU: {} Dice Score: {}".format(idx, iou, dice_score))
  average_iou += iou

grade = average_iou * 10

print("\nGrade is " + str(grade))

PASSING_GRADE = 60
if (grade>PASSING_GRADE):
  print("You passed!")
else:
  print("You failed. Please check your model and re-train")

## Save the Model

Once you're satisfied with the results, you can save your model's weights. (In the original course the Keras model file was uploaded to the Coursera grader; the PyTorch weights below are for your own use.)

In [ ]:
# Save the model you just trained
torch.save(model.state_dict(), "model.pt")

**Congratulations on completing this assignment on image segmentation!**